In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from pathlib import Path
import sys
import pandas as pd
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.pairs import PairDatasetBuilder
from src.benchmarks.pairwise_ols import PairwiseOLS

DATASETS = {
    "dunnhumby": {
        "path": ROOT / "data" / "dunnhumby" / "panel" / "dunnhumby_icdn_panel.parquet",
        "schema": PanelSchema(category="category", brand="brand", style="style"),
    },
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
}

SHORT = ["promo", "sin_52", "cos_52"]

def run_one(name, spec):
    panel = pd.read_parquet(spec["path"])
    panel = panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()
    print(f"\n=== {name} === {panel.shape}  "
          f"products={panel['product_code'].nunique()}  "
          f"stores={panel['store_code'].nunique()}")

    splitter = TemporalSplitter(period_col="week_id")
    train_raw, val_raw = splitter.single_split(panel, train_frac=0.8)

    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    print("controls:", len(SHORT), SHORT)

    builder = PairDatasetBuilder(SHORT)
    train_pairs = builder.build(train)
    val_pairs = builder.build(val)

    ols = PairwiseOLS(SHORT)
    own = ols.run_own(train, val)
    cross = ols.run_cross(train_pairs, val_pairs)

    print(f"own: {len(own)} products estimated  (of {train.product_code.nunique()})")
    print(f"cross: {len(cross)} pairs estimated  (of {train.product_code.nunique()**2 - train.product_code.nunique()})")
    if len(own):
        print("own  mean/min/max", own.own_elasticity.mean(), own.own_elasticity.min(), own.own_elasticity.max())
        print("own  median VIF", own.vif_log_price.median())
    if len(cross):
        print("cross mean/min/max", cross.cross_elasticity.mean(), cross.cross_elasticity.min(), cross.cross_elasticity.max())
        print("cross median VIF log_p_j", cross.vif_log_p_j.median())

    return own, cross

for name, spec in DATASETS.items():
    run_one(name, spec)


=== dunnhumby === (3216, 9)  products=10  stores=20
controls: 3 ['promo', 'sin_52', 'cos_52']
own: 10 products estimated  (of 10)
cross: 26 pairs estimated  (of 90)
own  mean/min/max -0.8703264499535417 -1.628025028315493 -0.2458109053367766
own  median VIF 2.5339023405951036
cross mean/min/max -0.1445769460744783 -4.5243274027846505 4.2542768199857655
cross median VIF log_p_j 1.6043226301901994

=== walmart === (47904, 7)  products=20  stores=10
controls: 3 ['promo', 'sin_52', 'cos_52']
own: 20 products estimated  (of 20)
cross: 380 pairs estimated  (of 380)
own  mean/min/max -0.6404019930809135 -2.6313206022811237 0.8434773239742677
own  median VIF 1.158572631618917
cross mean/min/max -0.09501590817959307 -3.239955785259081 2.496536508351011
cross median VIF log_p_j 1.0406757365104453

=== one_c === (12611, 7)  products=10  stores=20
controls: 3 ['promo', 'sin_52', 'cos_52']
own: 9 products estimated  (of 10)
cross: 58 pairs estimated  (of 90)
own  mean/min/max -0.11545217000916724 